In [ ]:
import kagglehub
path = kagglehub.dataset_download("pilarpieiro/tabular-dataset-ready-for-malicious-url-detection")
print(path)

Using Colab cache for faster access to the 'tabular-dataset-ready-for-malicious-url-detection' dataset.
/kaggle/input/tabular-dataset-ready-for-malicious-url-detection


In [ ]:
import numpy as np
import pandas as pd

import os
file_names = []
for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        file_names.append(os.path.join(dirname, filename))
print(file_names)

/kaggle/input/tabular-dataset-ready-for-malicious-url-detection/train_dataset.csv
/kaggle/input/tabular-dataset-ready-for-malicious-url-detection/test_dataset.csv
['/kaggle/input/tabular-dataset-ready-for-malicious-url-detection/train_dataset.csv', '/kaggle/input/tabular-dataset-ready-for-malicious-url-detection/test_dataset.csv']


In [ ]:
from matplotlib import pyplot

df_test = pd.read_csv(file_names[1])
df_train = pd.read_csv(file_names[0])

In [ ]:
def filter_columns_by_zero_frequency(
    df,
    binary_min_ones=0.01,
    binary_max_zeros=0.99,
    numeric_max_zeros=0.99,
    return_removed=False):

    numeric_cols = df.select_dtypes(include=["number"]).columns

    binary_cols = [
        c for c in numeric_cols
        if df[c].dropna().isin([0, 1]).all()]

    non_binary_numeric_cols = list(set(numeric_cols) - set(binary_cols))

    pct_ones = df[binary_cols].mean() if binary_cols else pd.Series(dtype=float)
    pct_zeros_binary = 1 - pct_ones

    pct_zeros_all = (df[numeric_cols] == 0).sum() / len(df)

    keep_binary = pct_ones[
        (pct_ones >= binary_min_ones) &
        (pct_zeros_binary <= binary_max_zeros)].index

    keep_non_binary = pct_zeros_all[
        pct_zeros_all.index.isin(non_binary_numeric_cols)
        & (pct_zeros_all <= numeric_max_zeros)].index

    keep_numeric = list(keep_binary) + list(keep_non_binary)

    removed_numeric = set(numeric_cols) - set(keep_numeric)

    keep_cols = list(keep_numeric) + list(df.columns.difference(numeric_cols))
    filtered_df = df[keep_cols]

    if return_removed:
        return filtered_df, sorted(list(removed_numeric))

    return filtered_df


In [ ]:
def normalize_decimals(df, eps=1e-9):
    df = df.copy()

    for col in df.select_dtypes(include=["number"]).columns:
        series = df[col]

        has_decimals = (np.abs(series - np.round(series)) > eps).any()

        if has_decimals:
            df.loc[:, col] = series * 10

    return df

In [ ]:
filtered_train, removed = filter_columns_by_zero_frequency(df_train, return_removed=True)
print("Removed columns by percent:", removed)
common_cols = filtered_train.columns.intersection(df_test.columns)

Removed columns by percent: ['path_count_no_of_embed', 'path_count_nonascii', 'path_count_pertwent', 'path_has_singlechardir', 'path_has_upperdir', 'pdomain_count_atrate', 'pdomain_count_digit', 'pdomain_count_hyphen', 'pdomain_count_non_alphanum', 'pdomain_len', 'url_count_atrate', 'url_count_hash', 'url_count_http', 'url_count_https', 'url_count_perc', 'url_count_semicolon', 'url_count_www', 'url_has_admin', 'url_has_client', 'url_has_ip', 'url_has_server']


train/val/test split (80/10/10)

In [ ]:
from sklearn.model_selection import train_test_split

df_test, df_val = train_test_split(df_test, test_size=0.5, random_state=42, shuffle=True)

filtered_test = df_test[common_cols].copy()
filtered_val = df_val[common_cols].copy()

In [ ]:
filtered_train = normalize_decimals(filtered_train)
filtered_test = normalize_decimals(filtered_test)
filtered_val = normalize_decimals(filtered_val)

In [ ]:
def get_binary_columns(df):
    binary_cols = []
    for col in df.select_dtypes(include=["number"]).columns:
        unique_vals = df[col].dropna().unique()
        if set(unique_vals).issubset({0, 1}):
            binary_cols.append(col)
    return binary_cols


def compute_quartile_bins(train_df):
    bins_dict = {}

    binary_cols = get_binary_columns(train_df)

    numeric_cols = [
        col for col in train_df.select_dtypes(include=["number"]).columns
        if col not in binary_cols]

    for col in numeric_cols:
        col_values = train_df[col].dropna()

        if col_values.nunique() <= 1:
            bins_dict[col] = "constant"
            continue

        q1, q2, q3 = col_values.quantile([0.25, 0.5, 0.75])

        if len({q1, q2, q3}) < 3:
            bins_dict[col] = "constant"
            continue

        bins_dict[col] = [-np.inf, q1, q2, q3, np.inf]

    return bins_dict


def apply_bins_replace(df, bins_dict):
    df = df.copy()

    for col, bins in bins_dict.items():

        if col not in df.columns:
            continue

        if bins == "constant":
            df[col] = 0
            continue

        df[col] = pd.cut(
            df[col],
            bins=bins,
            labels=[0, 1, 2, 3],
            duplicates="drop")

    return df

In [ ]:
bins_dict = compute_quartile_bins(filtered_train)

train_binned = apply_bins_replace(filtered_train, bins_dict)
val_binned   = apply_bins_replace(filtered_val,   bins_dict)
test_binned  = apply_bins_replace(filtered_test,  bins_dict)

print("Done! Non-binary numeric columns have been converted to 4-category features.")
print("Train cols:", len(filtered_train.columns), "->", len(train_binned.columns))
print("Val cols:  ", len(filtered_val.columns),   "->", len(val_binned.columns))
print("Test cols: ", len(filtered_test.columns),  "->", len(test_binned.columns))

Done! Non-binary numeric columns have been converted to 4-category features.
Train cols: 39 -> 39
Val cols:   39 -> 39
Test cols:  39 -> 39


In [ ]:
print("Train shape:", train_binned.shape)
print("Val shape:", test_binned.shape)
print("Test shape:", val_binned.shape)

Train shape: (6728848, 39)
Val shape: (841106, 39)
Test shape: (841107, 39)


In [ ]:
print(common_cols)


Index(['label', 'url_has_login', 'url_isshorted',
       'path_has_any_sensitive_words', 'tld_is_sus', 'url_len', 'url_entropy',
       'url_hamming_1', 'url_hamming_00', 'url_hamming_10', 'url_hamming_01',
       'url_hamming_11', 'url_2bentropy', 'url_3bentropy', 'url_count_dot',
       'url_count_hyphen', 'url_count_underscore', 'url_count_ques',
       'url_count_equal', 'url_count_amp', 'url_count_letter',
       'url_count_digit', 'url_count_sensitive_financial_words',
       'url_count_sensitive_words', 'url_nunique_chars_ratio', 'path_len',
       'path_count_no_of_dir', 'path_count_zero', 'path_count_lower',
       'path_count_upper', 'query_len', 'query_count_components', 'tld_len',
       'pdomain_min_distance', 'subdomain_len', 'subdomain_count_dot',
       'source', 'tld', 'url'],
      dtype='object')


In [ ]:
from sklearn.utils import resample

target_col = "label"

train_pos = train_binned[train_binned[target_col] == 1]
train_neg = train_binned[train_binned[target_col] == 0]

print("Before:")
print("Positives:", len(train_pos))
print("Negatives:", len(train_neg))

desired_pos = round((2/3) * len(train_neg))

train_pos_upsampled = resample(
    train_pos,
    replace=True,
    n_samples=desired_pos,
    random_state=42
)

train_inflated = pd.concat([train_neg, train_pos_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print("After:")
P = (train_inflated[target_col] == 1).sum()
N = (train_inflated[target_col] == 0).sum()
print("Positives:", P)
print("Negatives:", N)
print("Final positive ratio:", P / (P + N))


Before:
Positives: 1445673
Negatives: 5283175
After:
Positives: 3522117
Negatives: 5283175
Final positive ratio: 0.40000002271361357


In [ ]:
from sklearn.utils import resample

target_col = "label"

train_pos = filtered_val[filtered_val[target_col] == 1]
train_neg = filtered_val[filtered_val[target_col] == 0]

print("Before:")
print("Positives:", len(train_pos))
print("Negatives:", len(train_neg))

desired_pos = round((2/3) * len(train_neg))

train_pos_upsampled = resample(
    train_pos,
    replace=True,
    n_samples=desired_pos,
    random_state=42
)

val_inflated = pd.concat([train_neg, train_pos_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print("After:")
P = (val_inflated[target_col] == 1).sum()
N = (val_inflated[target_col] == 0).sum()
print("Positives:", P)
print("Negatives:", N)
print("Final positive ratio:", P / (P + N))


Before:
Positives: 722889
Negatives: 2641535
After:
Positives: 1761023
Negatives: 2641535
Final positive ratio: 0.3999999545718648


ALWAYS RUN LAST



In [ ]:
train_binned.to_csv("train_inflated_binned.csv", index=False)
val_binned.to_csv("val_binned.csv", index=False)
test_binned.to_csv("test_binned.csv", index=False)


In [ ]:
from google.colab import files

files.download("train_inflated_binned.csv")
files.download("val_binned.csv")
files.download("test_binned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>